# 

In [1]:
import sys
import os

# Get the parent directory and add it to sys.path
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
import pandas as pd
import os
from rdkit import Chem
from download_dataset import get_data_list
import selfies as sf

/miniconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mol_instruction_dataset = datasets.load_dataset(
    "zjunlp/Mol-Instructions",
    "Molecule-oriented Instructions",
    trust_remote_code=True,
)
qm9_data = mol_instruction_dataset['property_prediction']
qm9_homo_data = qm9_data.filter(
                lambda x: x["instruction"] in instructions_smol.filtering_template_homo
            )
qm9_lumo_data = qm9_data.filter(
                lambda x: x["instruction"] in instructions_smol.filtering_template_lumo
            )
qm9_homo_lumo_gap_data = qm9_data.filter(
                lambda x: x["instruction"] in instructions_smol.filtering_template_homo_lumo_gap
            )

In [3]:
def get_qm9_data_list(
        qm9_data,
        task,
        instruction_templates,
):
    list_tr_mol = []
    list_tr_label = []

    list_te_mol = []
    list_te_label = []

    from tqdm import tqdm
    iter_bar = tqdm(range(len(qm9_data)))
    for i in iter_bar:
        data_instance = qm9_data[i]
        selfies = data_instance['input']
        smiles = sf.decoder(selfies)
        mol = Chem.MolFromSmiles(smiles)
        label = data_instance['output']

        if "train" in data_instance['metadata']:
            list_tr_label.append(label)
            list_tr_mol.append(mol)
        elif "test" in data_instance['metadata']:
            list_te_label.append(label)
            list_te_mol.append(mol)
        elif "val" in data_instance['metadata']:
            pass
        else:
            print(data_instance)
            raise ValueError

    list_tr_data = get_data_list(
        list_mol=list_tr_mol,
        list_label=list_tr_label,
        task=task,
        instruction_templates=instruction_templates,
    )
    list_te_data = get_data_list(
        list_mol=list_te_mol,
        list_label=list_te_label,
        task=task,
        instruction_templates=instruction_templates,
    )
    print(len(list_tr_data), len(list_te_data))
    return list_tr_data, list_te_data

In [6]:
list_qm9_homo_tr_data, list_qm9_homo_te_data = get_qm9_data_list(
    qm9_data=qm9_homo_data,
    instruction_templates=instructions_smol.qm9_homo,
    task="molinst/homo",
)


 93%|█████████▎| 111866/120062 [00:55<00:04, 2013.95it/s]

In [ ]:

list_qm9_lumo_tr_data, list_qm9_lumo_te_data = get_qm9_data_list(
    qm9_data=qm9_lumo_data,
    instruction_templates=instructions_smol.qm9_lumo,
    task="molinst/lumo",
)

  0%|          | 0/120753 [00:00<?, ?it/s]

100%|██████████| 642/642 [00:00<00:00, 1987.47it/s]

120111 642


In [ ]:
list_qm9_homo_tr_data[0]

{'task': 'molinst/homo',
 'x': array([[5, 0, 4, 5, 3, 0, 2, 0, 0],
        [5, 0, 4, 5, 1, 0, 2, 0, 1],
        [6, 0, 3, 5, 0, 0, 2, 0, 1],
        [5, 0, 4, 5, 2, 0, 2, 0, 1],
        [5, 0, 4, 5, 1, 0, 2, 0, 1],
        [5, 0, 4, 5, 0, 0, 2, 0, 1],
        [5, 0, 4, 5, 3, 0, 2, 0, 0],
        [7, 0, 2, 5, 1, 0, 2, 0, 0]]),
 'edge_index': array([[0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 5, 7, 5, 1, 4, 2],
        [1, 0, 2, 1, 3, 2, 4, 3, 5, 4, 6, 5, 7, 5, 1, 5, 2, 4]]),
 'edge_attr': array([[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]]),
 'additional_x': array([[5, 0, 4, 5, 3, 0, 2, 0, 0],
        [5, 0, 4, 5, 1, 0, 2, 0, 1],
        [6, 0, 3, 5, 0, 0, 2, 0, 1],
        [5, 0, 4, 5, 2, 0, 2, 0, 1],
       

In [9]:
list_qm9_lumo_tr_data[0]

{'task': 'molinst/lumo',
 'x': array([[5, 0, 4, 5, 3, 0, 2, 0, 0],
        [6, 0, 3, 5, 1, 0, 1, 0, 0],
        [5, 0, 3, 5, 0, 0, 1, 0, 0],
        [7, 0, 1, 5, 0, 0, 1, 0, 0],
        [6, 0, 3, 5, 0, 0, 1, 0, 1],
        [5, 0, 4, 5, 2, 0, 2, 0, 1],
        [5, 0, 4, 5, 1, 0, 2, 0, 1],
        [5, 0, 4, 5, 2, 0, 2, 0, 1],
        [5, 0, 4, 5, 1, 0, 2, 0, 1]]),
 'edge_index': array([[0, 1, 1, 2, 2, 3, 2, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 4, 8, 6],
        [1, 0, 2, 1, 3, 2, 4, 2, 5, 4, 6, 5, 7, 6, 8, 7, 4, 8, 6, 8]]),
 'edge_attr': array([[0, 0, 0],
        [0, 0, 0],
        [0, 0, 1],
        [0, 0, 1],
        [1, 0, 1],
        [1, 0, 1],
        [0, 0, 1],
        [0, 0, 1],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]]),
 'additional_x': array([[5, 0, 4, 5, 3, 0, 2, 0, 0],
        [6, 0, 3, 5, 1, 0, 1, 0

In [10]:
list_qm9_homo_lumo_gap_tr_data, list_qm9_homo_lumo_gap_te_data = get_qm9_data_list(
    qm9_data=qm9_homo_lumo_gap_data,
    instruction_templates=instructions_smol.qm9_homo_lumo_gap,
    task="molinst/homo_lumo_gap",
)

100%|██████████| 661/661 [00:00<00:00, 2010.90it/s]

119940 661


In [11]:
list_qm9_homo_lumo_gap_tr_data[0]

{'task': 'molinst/homo_lumo_gap',
 'x': array([[7, 0, 1, 5, 0, 0, 1, 0, 0],
        [5, 0, 3, 5, 1, 0, 1, 0, 0],
        [5, 0, 4, 5, 1, 0, 2, 0, 1],
        [6, 0, 3, 5, 1, 0, 2, 0, 1],
        [5, 0, 4, 5, 1, 0, 2, 0, 1],
        [5, 0, 3, 5, 1, 0, 1, 0, 0],
        [7, 0, 1, 5, 0, 0, 1, 0, 0],
        [5, 0, 3, 5, 1, 0, 1, 0, 1],
        [5, 0, 3, 5, 1, 0, 1, 0, 1]]),
 'edge_index': array([[0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 4, 7, 7, 8, 8, 2],
        [1, 0, 2, 1, 3, 2, 4, 3, 5, 4, 6, 5, 7, 4, 8, 7, 2, 8]]),
 'edge_attr': array([[1, 0, 0],
        [1, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [0, 0, 0],
        [0, 0, 0]]),
 'additional_x': array([[7, 0, 1, 5, 0, 0, 1, 0, 0],
        [5, 0, 3, 5, 1, 0, 1, 0, 0],
        [5, 0, 4, 5, 1, 0, 2, 0, 1]

In [13]:
homo_dict = {
    "task": "homo",
    "train": list_qm9_homo_tr_data,
    "test": list_qm9_homo_te_data
}

lumo_dict = {
    "task": "lumo",
    "train": list_qm9_lumo_tr_data,
    "test": list_qm9_lumo_te_data
}

homo_lumo_gap_dict = {
    "task": "homo_lumo_gap",
    "train": list_qm9_homo_lumo_gap_tr_data,
    "test": list_qm9_homo_lumo_gap_te_data
}

for data_dict in [
    homo_dict,
    lumo_dict,
    homo_lumo_gap_dict
]:
    for split in ["train", "test"]:
        list_data = data_dict[split]
        task_name = data_dict["task"]
        
        dataset = datasets.Dataset.from_list(list_data)
        dataset.save_to_disk(
            f"/data/data/Mol-LLM-v8/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_{split}_molinst/{task_name}"
        )

Saving the dataset (1/1 shards): 100%|██████████| 642/642 [00:00<00:00, 3473.14 examples/s]


KeyboardInterrupt: 